# Chapter 6: Flow Matching
Chapter 5 ended at 0%. The diagnosis was capacity and multi-modality, not the idea. Here we replace the MLP denoiser with a **12.4M-param transformer action expert** and train it two ways -- rectified flow matching and DDPM -- on the same PushT data.

In [ ]:
!pip install torch torchvision numpy matplotlib transformers pillow
!pip install 'lerobot[dataset]' gym-pusht 'pymunk<7'

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/06_flow_matching

## The Idea: Straight Paths

Both methods learn to turn noise into actions. They differ in the *path* they take.

**Flow matching** defines a straight line between noise `x0` and data `x1`:

$$x_t = (1-t)\,x_0 + t\,x_1, \qquad \text{target} = x_1 - x_0$$

The target velocity is **constant along the path** -- it does not depend on `t`. So an Euler solver with a handful of steps integrates it almost exactly.

**DDPM** follows the curved path implied by its noise schedule. Same destination, but the velocity changes along the way, so coarse integration accumulates error. That single difference is most of this chapter's result.

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

x0 = torch.tensor([-1.6, -0.9])      # noise sample
x1 = torch.tensor([1.3, 1.1])        # target action
ts = torch.linspace(0, 1, 25)

# Flow matching: straight-line interpolation, constant velocity x1 - x0.
fm_path = torch.stack([(1 - t) * x0 + t * x1 for t in ts])

# DDPM: the cosine schedule's alpha_bar sets how much signal survives at each t.
#   x_t = sqrt(alpha_bar) * x1 + sqrt(1 - alpha_bar) * x0
alpha_bar = torch.cos((ts + 0.008) / 1.008 * math.pi / 2) ** 2
alpha_bar = alpha_bar / alpha_bar[0]
ddpm_path = torch.stack([
    ab.sqrt() * x1 + (1 - ab).sqrt() * x0 for ab in reversed(alpha_bar)
])

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(*fm_path.T, "o-", ms=3, label="Flow matching (straight)")
ax.plot(*ddpm_path.T, "s-", ms=3, label="DDPM (curved)")
ax.scatter(*x0, s=120, marker="X", zorder=5, label="noise $x_0$")
ax.scatter(*x1, s=120, marker="*", zorder=5, label="action $x_1$")
ax.set_title("Path from noise to action")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

print("Flow matching velocity is constant along the path:")
v = fm_path[1:] - fm_path[:-1]
print(f"  step-to-step variation: {(v - v.mean(0)).abs().max():.2e}")
d = ddpm_path[1:] - ddpm_path[:-1]
print(f"DDPM step size varies by: {d.norm(dim=1).max() / d.norm(dim=1).min():.1f}x")

## The Action Expert

A transformer that denoises a *chunk* of actions. Each of its 6 blocks is **self-attention over the action chunk -> cross-attention into visual context -> FFN**.

The cross-attention is interleaved on purpose: every block re-reads the observation, rather than conditioning once at the input and hoping the signal survives 6 layers.

In [ ]:
from action_expert import build_action_expert

model = build_action_expert(
    action_dim=2, chunk_size=16, state_dim=2,
    d_model=384, nhead=6, num_layers=6, ffn_dim=1024, dropout=0.1,
)
print(f"Parameters: {model.count_parameters() / 1e6:.1f}M")
print(model.blocks[0])

## Data

Same PushT cache pipeline as Chapter 5, under a `ch06_` prefix.

In [ ]:
from flow_matching import (
    extract_and_cache, split_by_episode, compute_norm_stats,
    RobotDataset, PRESETS, DEVICE,
)

cfg = PRESETS["pusht"]
data = extract_and_cache(
    repo_id=cfg["repo_id"], image_key=cfg["image_key"],
    state_key=cfg["state_key"], action_key=cfg["action_key"],
    cache_prefix="ch06_pusht", skip_embeddings=True,
)

train_mask, val_mask = split_by_episode(data["episode_indices"])
action_stats = compute_norm_stats(data["actions"][train_mask])
state_stats = compute_norm_stats(data["states"][train_mask])
norm_actions = action_stats.normalize(data["actions"])
norm_states = state_stats.normalize(data["states"])

print(f"device: {DEVICE}")
print(f"cached keys: {list(data)}")
print(f"cls embeddings {tuple(data['cls_embeddings'].shape)}  (SigLIP CLS per frame)")
print(f"train {int(train_mask.sum())} frames | val {int(val_mask.sum())} frames")

## Train Flow Matching vs DDPM

Two methods x two chunk sizes (K=4, K=16). Both use the *same* transformer -- only the training objective and sampler differ, which is what makes the comparison fair.

EMA (decay 0.999) is used for inference; generative models sample noticeably better from averaged weights.

We use fewer epochs than the published 200-epoch run so this finishes in one session.

In [ ]:
from pathlib import Path
from flow_matching import (
    FlowMatchingTrainer, DDPMTransformerTrainer, train_model, METHODS, CHUNK_SIZES,
)

EPOCHS = 40           # published run uses 200 -- see README
ckpt_dir = Path("checkpoints"); ckpt_dir.mkdir(exist_ok=True)
runs = {}

for method in METHODS:
    for cs in CHUNK_SIZES:
        name = f"{method}_K{cs}"
        print(f"\n--- {name} ---")

        net = build_action_expert(
            action_dim=cfg["action_dim"], chunk_size=cs, state_dim=cfg["state_dim"],
            d_model=cfg["d_model"], nhead=cfg["nhead"], num_layers=cfg["num_layers"],
            ffn_dim=cfg["ffn_dim"], dropout=cfg["dropout"],
        )
        if method == "flow_matching":
            trainer = FlowMatchingTrainer(net, inference_steps=cfg["fm_inference_steps"])
        else:
            trainer = DDPMTransformerTrainer(net, inference_steps=cfg["ddpm_inference_steps"])

        mk = lambda m: RobotDataset(data["cls_embeddings"][m], norm_states[m],
                                    norm_actions[m], data["episode_indices"][m],
                                    chunk_size=cs)
        out = train_model(trainer, mk(train_mask), mk(val_mask), epochs=EPOCHS,
                          batch_size=cfg["batch_size"], lr=cfg["lr"],
                          ckpt_path=ckpt_dir / f"ch06_pusht_{name}.pt", device=DEVICE)
        runs[name] = {"trainer": trainer, "ema": out["ema"], "history": out["history"], "cs": cs}

## Validation Loss -- and Why It Misleads

Watch for the trap: **DDPM reaches lower validation loss than flow matching**, then samples worse. The loss measures how well the network predicts its own training target (noise vs velocity); those are different quantities on different scales and are not comparable across methods. Only rollout tells you which sampler works.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, r in runs.items():
    if r["history"].get("val_loss"):
        ax.plot(r["history"]["val_loss"], label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Val loss"); ax.set_yscale("log")
ax.set_title("Validation loss -- not comparable across objectives")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Live Rollout: Coverage and Latency

`avg_coverage` is the fraction of the target region the T-block was pushed into -- a graded score, more informative than binary success when no policy is near 100%.

One caveat, stated up front: at the reduced epoch count above, and over only `eval_episodes` rollouts, **coverage is noisy enough that DDPM can beat flow matching on any given run**. The separation reported below is from the full 200-epoch run. Do not read a single short run as the result -- that is the same mistake Chapter 5 warns about, one level up.

In [ ]:
from flow_matching import evaluate_live_pusht

live = {}
for name, r in runs.items():
    steps = (cfg["fm_inference_steps"] if name.startswith("flow_matching")
             else cfg["ddpm_inference_steps"])
    live[name] = evaluate_live_pusht(
        r["trainer"], r["ema"], state_stats, action_stats, r["cs"],
        n_episodes=cfg["eval_episodes"], num_inference_steps=steps, device=DEVICE,
    )
    print(f"{name:<20} coverage={live[name]['avg_coverage']*100:5.1f}%  "
          f"inference={live[name]['avg_inference_ms']:5.1f} ms")

names = list(live)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(names, [live[n]["avg_coverage"] * 100 for n in names])
axes[0].set_ylabel("Coverage (%)"); axes[0].set_title("Rollout coverage")
axes[1].bar(names, [live[n]["avg_inference_ms"] for n in names], color="tab:orange")
axes[1].set_ylabel("Inference (ms)"); axes[1].set_title("Sampling latency")
for a in axes:
    a.tick_params(axis="x", rotation=30, labelsize=8); a.grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## How Few Euler Steps Can We Get Away With?

Flow matching's straight path is the reason this question has a good answer. Fewer steps means lower latency, which is what makes a VLA controllable at 30 Hz.

In [ ]:
import time

fm = next((n for n in runs if n.startswith("flow_matching") and runs[n]["cs"] == 16), None)
r = runs[fm]
print(f"Ablating {fm}\n")

ablation = {}
for steps in [3, 5, 10, 20]:
    m = evaluate_live_pusht(r["trainer"], r["ema"], state_stats, action_stats, r["cs"],
                            n_episodes=cfg["eval_episodes"],
                            num_inference_steps=steps, device=DEVICE)
    ablation[steps] = m
    print(f"  steps={steps:<3} coverage={m['avg_coverage']*100:5.1f}%  "
          f"inference={m['avg_inference_ms']:5.1f} ms")

fig, ax = plt.subplots(figsize=(6, 4))
ks = list(ablation)
ax.plot(ks, [ablation[k]["avg_coverage"] * 100 for k in ks], "o-", label="Coverage (%)")
ax2 = ax.twinx()
ax2.plot(ks, [ablation[k]["avg_inference_ms"] for k in ks], "s--", color="tab:orange",
         label="Inference (ms)")
ax.set_xlabel("Euler steps"); ax.set_ylabel("Coverage (%)")
ax2.set_ylabel("Inference (ms)")
ax.set_title("Quality vs latency")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## What We Learned

**Flow matching beats DDPM by ~5x coverage at half the latency** (29.0% vs 5.3% on the full run, 13 ms vs 28 ms). Same transformer, same data, same compute -- only the path differs. Straight paths integrate cleanly with 10 Euler steps; curved ones do not.

**DDPM wins on validation loss and loses on the task.** Do not rank generative policies by their training loss. Noise-prediction MSE and velocity MSE are different targets; the number that matters comes from the environment.

**Capacity had to come *down*, not up.** The first version of this model was 96.3M params and overfit 22K frames catastrophically. 12.4M plus dropout=0.1 fixed it -- comparable to Diffusion Policy's ~16M UNet. Chapter 5's MLP failed for too little capacity; the naive fix overshot.

**10 steps is the sweet spot.** Coverage climbs to 10 steps, dips at 20, and only exceeds it again at 50 steps and 63 ms -- far too slow for a 30 Hz control loop.

**Coverage ~29% is not success**, and we are not going to dress it up. 206 demonstrations is not enough. But the policy is genuinely pushing the block toward the target, which nothing in Chapter 5 did.

**Next:** Chapter 7 assembles the whole thing -- SigLIP + SmolLM2 + this action expert -- into a SmolVLA-like model trained on real SO100 robot data.